# Finding Parkinson's disease taxonomic analyses

In this notebook we aim to imitate the analyses in ["`ABaCo` demo: Parkinson’s disease gut microbiome"](https://mona-abaco.readthedocs.io/en/latest/tutorial/demo-parkinson.html) where the aim was to "integrate the 9 studies while preserving key distinctions from the two patient states (Parkinson’s v.s. Healthy)."

```{margin}
After clicking the "Activate Notebook" button you can run the cells in this browser. Alternatively, you can also click on the 🚀 to launch in colab or binder. 
```
<button title="Make live" style="display:inline-flex;align-items:center;gap:0.4rem;padding:0.5rem 1rem;border:0;border-radius:20px;background:linear-gradient(135deg,#0f766e,#14b8a6);color:white;cursor:pointer;font-size:1rem;" class="thebe-button" onclick="initThebeSBT()">Activate Notebook</button>

---

In [ ]:
# uncomment if colab
# !pip install mgnipy

In [1]:
import logging 
logging.basicConfig(level=logging.WARNING)

## Searching for studies

To start we configure our MGnipy client and access the MGnify API Studies resource. 

We will filter our query to studies of the gut microbiome that mention "parkinson"s disease. 

We can preview the resulting query urls via `.explain()`

In [2]:
from mgnipy import MGnipy 

mg = MGnipy(cache_dir='downloads')

pd_studies = mg.studies(
    search='parkinson',
    biome_lineage='root:Host-associated:Human:Digestive system:Large intestine:Fecal',
)

pd_studies.explain()

https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AHost-associated%3AHuman%3ADigestive+system%3ALarge+intestine%3AFecal&search=parkinson&page=1


looks good. we can proceed with actually executing the list query/queries via .get(). To enrich our list of studies with metadata details we can do this in bulk using `.enrich_details()` or asynchronously via `.aenrich_details()`

In [3]:
# populate study list
pd_studies.get()
# enrich studies with metadta
await pd_studies.aenrich_details()

# or even save to file if you prefer
study_meta = pd_studies.details_df(expand_nested_dicts=True)

# check it out 
study_meta.head()

Enriching study details: 100%|██████████| 8/8 [00:03<00:00,  2.01it/s]


,accession,ena_accessions,title,updated_at,downloads,first_accession,metadata__study_name,metadata__center_name,metadata__study_title,metadata__study_accession,metadata__study_description,metadata__secondary_study_accession,biome__biome_name,biome__lineage
0,MGYS00006759,"[ERP146353, PRJEB61255]",EMG produced TPA metagenomics assembly of PRJN...,2026-05-28T15:47:02.660000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP146353,NaN,NaN,NaN,NaN,NaN,NaN,Fecal,root:Host-associated:Human:Digestive system:La...
1,MGYS00005755,"[PRJNA510730, SRP173877]",Microbiota composition of Parkinson's disease ...,2026-05-06T11:35:50.490000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",SRP173877,NaN,NaN,NaN,NaN,NaN,NaN,Fecal,root:Host-associated:Human:Digestive system:La...
2,MGYS00001650,"[ERP004264, PRJEB4927]",Alterations of the Fecal Microbiome in Parkins...,2026-05-28T15:47:02.598000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP004264,Fecal Microbiome in Parkinson's Disease,Institute of Biotechnology;University of Helsi...,Alterations of the Fecal Microbiome in Parkins...,PRJEB4927,"In the course of Parkinson’s disease (PD), the...",ERP004264,Fecal,root:Host-associated:Human:Digestive system:La...
3,MGYS00005601,"[ERP113090, PRJEB30615]",Identification of Intestinal Bacterial Taxa wi...,2026-05-28T15:47:01.054000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP113090,NaN,NaN,NaN,NaN,NaN,NaN,Fecal,root:Host-associated:Human:Digestive system:La...
4,MGYS00006121,"[ERP142200, PRJEB57228]",Dietary intervention of people with Parkinson'...,2026-05-28T15:47:01.432000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP142200,NaN,NaN,NaN,NaN,NaN,NaN,Fecal,root:Host-associated:Human:Digestive system:La...


## Using `MGazine` to explore the study datasets

we can access the mgazine of datasets via `.datasets` attribute. The study details we retrieved above will also be passed on to the mgazine

In [4]:
# access mgazine
mz = pd_studies.datasets

# take a look
print(mz)

MGazine containing:
- MGnify pipeline versions: ['v3', 'v4_1', 'v5', 'v6']
- Number of downloads: 72
- Short descriptions: ['Complete GO annotation',
 'DwC-Ready summary of 16S-V3-V4 ASV taxonomies using -PR2 as ref DB',
 'DwC-Ready summary of 16S-V3-V4 ASV taxonomies using -SILVA as ref DB',
 'DwC-Ready summary of closed-ref taxonomies using ITSoneDB as ref DB',
 'DwC-Ready summary of closed-ref taxonomies using PR2 as ref DB',
 'DwC-Ready summary of closed-ref taxonomies using SILVA-LSU as ref DB',
 'DwC-Ready summary of closed-ref taxonomies using SILVA-SSU as ref DB',
 'GO slim annotation',
 'InterPro matches',
 'Phylum level taxonomies',
 'Phylum level taxonomies LSU',
 'Phylum level taxonomies SSU',
 'Summary of DADA2-PR2 taxonomies',
 'Summary of DADA2-SILVA taxonomies',
 'Summary of ITSoneDB taxonomies',
 'Summary of PR2 taxonomies',
 'Summary of SILVA-LSU taxonomies',
 'Summary of SILVA-SSU taxonomies',
 'Taxonomic assignments',
 'Taxonomic assignments LSU',
 'Taxonomic assign

For the ABaCo demo we will use the taxonomic analyses and we will use v4 onwards due to differences in pipeline versions and specifically SILVA databases that were used for the taxonomic analysis

In [5]:
# can add magazines
mz_taxa = mz['Summary of SILVA-SSU taxonomies'] + mz.v5['Taxonomic assignments SSU']   

# print still works
print(mz_taxa)

# studies details are preserved
import pandas as pd
display(pd.DataFrame(mz_taxa.studies_details))

'Summary of SILVA-SSU taxonomies' used for `long_short_mapping` determination and caching.


MGazine Curation TaxaMGazine containing:
- MGnify pipeline versions: ['v6']
- Number of downloads: 3
- Short descriptions: ['Summary of SILVA-SSU taxonomies']
-----------------------
Next steps: Use `.load()` to initialize.

MGazine Curation TaxaMGazine containing:
- MGnify pipeline versions: ['v5']
- Number of downloads: 4
- Short descriptions: ['Taxonomic assignments SSU']
-----------------------
Next steps: Use `.load()` to initialize.

MGazine Curation TaxaMGazine containing:
- MGnify pipeline versions: ['v5', 'v6']
- Number of downloads: 7
- Short descriptions: ['Summary of SILVA-SSU taxonomies', 'Taxonomic assignments SSU']
-----------------------
Next steps: Use `.load()` to initialize.

MGazine Curation TaxaMGazine containing:
- MGnify pipeline versions: ['v5', 'v6']
- Number of downloads: 7
- Short descriptions: ['Summary of SILVA-SSU taxonomies', 'Taxonomic assignments SSU']



,accession,ena_accessions,title,biome,updated_at,downloads,metadata,first_accession
0,MGYS00006759,"[ERP146353, PRJEB61255]",EMG produced TPA metagenomics assembly of PRJN...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-28T15:47:02.660000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",{},ERP146353
1,MGYS00005755,"[PRJNA510730, SRP173877]",Microbiota composition of Parkinson's disease ...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-06T11:35:50.490000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",{},SRP173877
2,MGYS00001650,"[ERP004264, PRJEB4927]",Alterations of the Fecal Microbiome in Parkins...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-28T15:47:02.598000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",{'study_name': 'Fecal Microbiome in Parkinson'...,ERP004264
3,MGYS00005601,"[ERP113090, PRJEB30615]",Identification of Intestinal Bacterial Taxa wi...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-28T15:47:01.054000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",{},ERP113090
4,MGYS00006121,"[ERP142200, PRJEB57228]",Dietary intervention of people with Parkinson'...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-28T15:47:01.432000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",{},ERP142200
5,MGYS00005129,"[ERP109659, PRJEB27564]",Gut microbiota in Parkinson's disease: tempora...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-06T12:25:31.349000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",{'study_name': 'Parkinson's disease gut microb...,ERP109659
6,MGYS00006760,"[ERP148661, PRJEB63522]",EMG produced TPA metagenomics assembly of PRJN...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-28T15:47:02.672000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",{},ERP148661
7,MGYS00005130,"[ERP112853, PRJEB30401]",Gut Microbiome Alterations Drive Distinct Meta...,"{'biome_name': 'Fecal', 'lineage': 'root:Host-...",2026-05-06T10:02:48.163000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",{'study_name': 'Gut Microbiome and Parkinson's...,ERP112853


## (Lazy)Loading into one taxonomic dataset

In [6]:
# lazyload the mgnify taxanomic assignments datasets
mz_taxa.load()

# calling to_pandas or to_polars will collect the data and return a dataframe
mz_taxa.to_polars().head()

TaxaMGazine loaded with 7 datasets. 
Cached runs results: 0 of total 1828.


taxonomy,SRR11321049,SRR11321050,SRR11321051,SRR11321052,SRR11321053,SRR11321054,SRR11321055,SRR11321056,SRR11321057,SRR11321058,SRR11321059,SRR11321060,SRR11321061,SRR11321062,SRR11321063,SRR11321064,SRR11321065,SRR11321066,SRR11321067,SRR11321068,SRR11321069,SRR11321070,SRR11321071,SRR11321072,SRR11321073,SRR11321074,SRR11321075,SRR11321076,SRR11321077,SRR11321078,SRR11321079,SRR11321080,SRR11321081,SRR11321082,SRR11321083,SRR11321084,…,ERZ19289368,ERZ19289388,ERZ19289398,ERZ19289428,ERZ19289448,ERZ19289468,ERZ19289478,ERZ19289498,ERZ19289528,ERZ19289578,ERZ19289588,ERZ19289598,ERZ19289608,ERZ19289618,ERZ19289628,ERZ19289319,ERZ19289329,ERZ19289339,ERZ19289349,ERZ19289359,ERZ19289369,ERZ19289389,ERZ19289399,ERZ19289409,ERZ19289419,ERZ19289429,ERZ19289449,ERZ19289469,ERZ19289479,ERZ19289489,ERZ19289539,ERZ19289549,ERZ19289579,ERZ19289589,ERZ19289599,ERZ19289619,ERZ19289629
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""sk__Archaea""",0,0,0,0,0,0,0,0,3,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,3,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""sk__Archaea;k__;p__Candidatus_…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""sk__Archaea;k__;p__Candidatus_…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""sk__Archaea;k__;p__Candidatus_…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""sk__Archaea;k__;p__Candidatus_…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


## Enriching with metadata

### taxonomic info

In [7]:
mz_taxa.taxonomic_metadata()

,Superkingdom,Kingdom,Phylum,Class,Order,Family,Genus,Species
0,Archaea,NA,NA,NA,NA,NA,NA,NA
1,Archaea,NA,Candidatus_Thermoplasmatota,Thermoplasmata,NA,NA,NA,NA
2,Archaea,NA,Candidatus_Thermoplasmatota,Thermoplasmata,Methanomassiliicoccales,NA,NA,NA
3,Archaea,NA,Candidatus_Thermoplasmatota,Thermoplasmata,Methanomassiliicoccales,Methanomassiliicoccaceae,Methanomassiliicoccus,NA
4,Archaea,NA,Candidatus_Thermoplasmatota,Thermoplasmata,Methanomassiliicoccales,Methanomassiliicoccaceae,Methanomassiliicoccus,Candidatus_Methanomassiliicoccus_intestinalis
...,...,...,...,...,...,...,...,...
2912,Bacteria,NA,Bacteroidetes,Chitinophagia,Chitinophagales,NA,NA,NA
2913,Bacteria,NA,Firmicutes,Clostridia,Clostridiales,Lachnospiraceae,Butyrivibrio,Butyrivibrio_crossotus
2914,Bacteria,NA,Firmicutes,Clostridia,Clostridiales,Clostridiaceae,Massilioclostridium,Massilioclostridium_coli
2915,Bacteria,NA,Bacteroidetes,Bacteroidia,Bacteroidales,Porphyromonadaceae,Sanguibacteroides,Sanguibacteroides_justesenii


if we take a look at the metadata it will be empty

In [8]:
mz_taxa.metadata().head()

""
accession
SRR11321049
SRR11321050
SRR11321051
SRR11321052
SRR11321053


In [9]:
mz_taxa.to_anndata()

/Users/anglup/.pyenv/versions/3.11.7/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


AnnData object with n_obs × n_vars = 2917 × 1828
    obs: 'Superkingdom', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species'

however we can add additional metadata that we collected manually, or taxacurator can help some

In [13]:
# getting some runs metdata, can run this cell multi times
await mz_taxa.aenrich_runs(limit=200)

# check it out
df_runs = pd.DataFrame(mz_taxa.runs_details)
print(df_runs.shape)
display(df_runs.head())

Enriching runs:  44%|████▍     | 800/1828 [00:07<00:35, 28.56it/s]  

(800, 16)


,experiment_type,instrument_model,instrument_platform,sample,study,accession,sample_accession,study_accession,updated_at,run_accession,reads_study_accession,assembly_study_accession,assembler_name,assembler_version,metadata,status
0,Amplicon,NaN,NaN,"{'accession': 'SAMN14518471', 'ena_accessions'...","{'accession': 'MGYS00005755', 'ena_accessions'...",SRR11321050,SAMN14518471,MGYS00005755,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Amplicon,NaN,NaN,"{'accession': 'SAMN14518507', 'ena_accessions'...","{'accession': 'MGYS00005755', 'ena_accessions'...",SRR11321059,SAMN14518507,MGYS00005755,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Amplicon,NaN,NaN,"{'accession': 'SAMN14518511', 'ena_accessions'...","{'accession': 'MGYS00005755', 'ena_accessions'...",SRR11321055,SAMN14518511,MGYS00005755,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Amplicon,NaN,NaN,"{'accession': 'SAMN14518470', 'ena_accessions'...","{'accession': 'MGYS00005755', 'ena_accessions'...",SRR11321051,SAMN14518470,MGYS00005755,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Amplicon,NaN,NaN,"{'accession': 'SAMN14518509', 'ena_accessions'...","{'accession': 'MGYS00005755', 'ena_accessions'...",SRR11321057,SAMN14518509,MGYS00005755,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
mz_taxa.enrich_biosamples(limit=10, incl_ena=True)

# check it out
df_biosam = pd.DataFrame(mz_taxa.biosamples_details)
print(df_biosam.shape)
display(df_biosam.head())

Enriching biosamples:   1%|          | 20/1828 [00:12<36:18,  1.20s/it]

(20, 33)


,GivenID,RunID,StudyID,SampleID,seq_meth,decimalLatitude,decimalLongitude,depth,center_name,temperature,...,NCBI submission package,collection date,geo loc name,host,host_phenotype,isolation source,lat lon,organism,status,title
0,SRR11321049,SRR11321049,SRP173877,SAMN14518472,Illumina MiSeq,41.0,12.0,NA,University of Rome Tor Vergata,NA,...,Metagenome.environmental.1.0,2019-12,Italy: Rome,Homo sapiens,Parkinson's Disease,feaces,41 N 12 E,metagenome,PD 152,Metagenome or environmental sample from metage...
1,SRR11321050,SRR11321050,SRP173877,SAMN14518471,Illumina MiSeq,41.0,12.0,NA,University of Rome Tor Vergata,NA,...,Metagenome.environmental.1.0,2019-12,Italy: Rome,Homo sapiens,Parkinson's Disease,feaces,41 N 12 E,metagenome,PD 175,Metagenome or environmental sample from metage...
2,SRR11321051,SRR11321051,SRP173877,SAMN14518470,Illumina MiSeq,41.0,12.0,NA,University of Rome Tor Vergata,NA,...,Metagenome.environmental.1.0,2019-12,Italy: Rome,Homo sapiens,Parkinson's Disease,feaces,41 N 12 E,metagenome,PD 156,Metagenome or environmental sample from metage...
3,SRR11321052,SRR11321052,SRP173877,SAMN14518469,Illumina MiSeq,41.0,12.0,NA,University of Rome Tor Vergata,NA,...,Metagenome.environmental.1.0,2019-12,Italy: Rome,Homo sapiens,Parkinson's Disease,feaces,41 N 12 E,metagenome,PD 177,Metagenome or environmental sample from metage...
4,SRR11321053,SRR11321053,SRP173877,SAMN14518468,Illumina MiSeq,41.0,12.0,NA,University of Rome Tor Vergata,NA,...,Metagenome.environmental.1.0,2019-12,Italy: Rome,Homo sapiens,Parkinson's Disease,feaces,41 N 12 E,metagenome,PD 158,Metagenome or environmental sample from metage...


In [ ]:
# PICK UP HERE

In [ ]:
# from abaco.dataloader import DataPreprocess, one_hot_encoding
# # Load Parkinson's disease dataset
# path_to_dataset = 'data/dataset_parkinson.csv'
# batch_col = "study_code"
# bio_col = "phenotype"
# id_col = "samples"

# # Convert data path into compatible pd.DataFrame
# df_parkinson = DataPreprocess(
#     path_to_dataset,
#     factors = [
#         id_col,
#         batch_col,
#         bio_col
#     ]
# ).dropna()

# # see if there are 3 categorical and n numeric columns (should be an extra column for location)
# print(df_parkinson.info())